# Forward to the Past — Enrichissement du catalogue

Ce notebook complète le catalogue de films construit dans le notebook de création du catalogue. Il consolide les acteurs et réalisateurs à partir des données IMDb et des compléments disponibles, puis ajoute les descriptions françaises issues de TMDB ou d’une traduction.

Le catalogue initial reste inchangé. Le résultat enrichi est exporté dans `data/processed/dataset_catalogue_filtres.csv`.

Exécuter les cellules dans l’ordre depuis un kernel vierge. Les fichiers intermédiaires, s’ils sont utilisés, doivent avoir été générés pour le même catalogue et les mêmes suppléments.

## 0 — Imports et configuration

Les chemins, les options de lecture et les règles de sélection des personnes sont regroupés ici.

### 0.1 — Imports

In [1]:
import sys
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

### 0.2 — Configuration technique

La racine du dépôt correspond au parent du répertoire de travail (exécution locale depuis `notebooks/`). Adapter les chemins si nécessaire.

`UTILISER_INTERIM` permet de choisir entre le chargement des CSV intermédiaires (`True`) et la réduction des archives IMDb brutes (`False`). En mode intermédiaire, les fichiers doivent avoir été préparés avec `src/exporter_sources_reduites.py`. Le notebook ne reconstruit pas ces fichiers automatiquement.

In [ ]:
UTILISER_INTERIM = False

RACINE_PROJET = Path.cwd().parent
DOSSIER_INTERIM = RACINE_PROJET / 'data/interim'
DOSSIER_SUPPLEMENTS = RACINE_PROJET / 'data/supplements'
DOSSIER_PROCESSED = RACINE_PROJET / 'data/processed'

CHEMIN_CATALOGUE = DOSSIER_PROCESSED / 'catalogue.csv'
CHEMIN_TITLE_PRINCIPALS = DOSSIER_INTERIM / 'title.principals.csv'
CHEMIN_NAME_BASICS = DOSSIER_INTERIM / 'name.basics.csv'
CHEMIN_SUPPLEMENT_PRINCIPALS = DOSSIER_SUPPLEMENTS / 'title.principals.csv'
CHEMIN_DESCRIPTIONS_FR = DOSSIER_SUPPLEMENTS / 'film_description_enrichi.csv'
CHEMIN_CATALOGUE_ENRICHI = DOSSIER_PROCESSED / 'dataset_catalogue_filtres.csv'

OPTIONS_CSV = {
    'dtype': 'string',
    'na_values': [r'\N'],
}

sys.path.insert(0, str(RACINE_PROJET))
from src import reduction_sources as rs

### 0.3 — Configuration métier

Les acteurs, actrices et personnes apparaissant dans leur propre rôle sont regroupés sous `acteur`. Les réalisateurs constituent un groupe distinct. Les images et sons d’archives et les autres métiers ne sont pas retenus.

In [3]:
CATEGORIES_ACTEURS = ['actor', 'actress', 'self']
CATEGORIES_RETENUES = [*CATEGORIES_ACTEURS, 'director']

## 1 — Chargement des sources

Le catalogue fournit les identifiants des films à enrichir. Les relations IMDb, les suppléments éventuels et les noms associés sont chargés avant la consolidation. Les descriptions françaises sont chargées séparément.

### 1.1 — Catalogue initial

Charger le catalogue filtré produit par le notebook de création du catalogue.

In [4]:
print("Chargement du catalogue…")

catalogue = pd.read_csv(
    CHEMIN_CATALOGUE,
    **OPTIONS_CSV,
)
ids_films = catalogue['tconst'].dropna().unique()

print(f"[OK] {len(catalogue):,} films chargés")

Chargement du catalogue…
[OK] 7,356 films chargés


### 1.2 — Suppléments IMDb

Le fichier `data/supplements/title.principals.csv` est facultatif. Il contient les relations film-personne ajoutées manuellement, au même format que la source IMDb. Seules les relations associées aux films du catalogue sont conservées. En son absence, la consolidation s’appuie uniquement sur IMDb.

In [5]:
print("Chargement des suppléments…")

if CHEMIN_SUPPLEMENT_PRINCIPALS.is_file():
    supplement_principals = pd.read_csv(
        CHEMIN_SUPPLEMENT_PRINCIPALS,
        **OPTIONS_CSV,
    )
    supplement_principals = supplement_principals.loc[
        supplement_principals['tconst'].isin(ids_films)
    ].copy()
    print(f"[OK] Title Principals : {len(supplement_principals):,} lignes")
else:
    supplement_principals = pd.DataFrame(
        columns=['tconst', 'ordering', 'nconst', 'category', 'job', 'characters'],
        dtype='string',
    )
    print("[IGNORÉ] Aucun supplément Title Principals")

Chargement des suppléments…
[OK] Title Principals : 18 lignes


### 1.3 — IMDb Title Principals

Charger les relations film-personne depuis le fichier intermédiaire ou réduire l’archive brute aux films du catalogue. Les compléments manuels ne sont pas encore intégrés à cette table.

In [6]:
debut = perf_counter()

print("Chargement de Title Principals…")

if UTILISER_INTERIM:
    print("Mode : fichier intermédiaire")
    title_principals = pd.read_csv(
        CHEMIN_TITLE_PRINCIPALS,
        **OPTIONS_CSV,
    )
else:
    print("Mode : réduction de la source brute")
    title_principals = rs.reduire_title_principals(ids_films)

print(
    f"[OK] {len(title_principals):,} lignes chargées | "
    f"{perf_counter() - debut:.1f} s"
)

Chargement de Title Principals…
Mode : réduction de la source brute
  Bloc 10 | 2,000,000 lignes lues | 79,624 retenues | 2.3 s
  Bloc 20 | 4,000,000 lignes lues | 100,102 retenues | 4.6 s
  Bloc 30 | 6,000,000 lignes lues | 114,079 retenues | 6.8 s
  Bloc 40 | 8,000,000 lignes lues | 114,079 retenues | 8.8 s
  Bloc 50 | 10,000,000 lignes lues | 116,757 retenues | 10.9 s
  Bloc 60 | 12,000,000 lignes lues | 119,649 retenues | 13.0 s
  Bloc 70 | 14,000,000 lignes lues | 120,658 retenues | 14.9 s
  Bloc 80 | 16,000,000 lignes lues | 121,472 retenues | 16.9 s
  Bloc 90 | 18,000,000 lignes lues | 122,155 retenues | 18.9 s
  Bloc 100 | 20,000,000 lignes lues | 122,994 retenues | 20.9 s
  Bloc 110 | 22,000,000 lignes lues | 124,400 retenues | 22.9 s
  Bloc 120 | 24,000,000 lignes lues | 125,570 retenues | 24.9 s
  Bloc 130 | 26,000,000 lignes lues | 126,435 retenues | 26.9 s
  Bloc 140 | 28,000,000 lignes lues | 127,434 retenues | 29.0 s
  Bloc 150 | 30,000,000 lignes lues | 128,548 retenues

### 1.4 — IMDb Name Basics

Charger les noms associés aux relations. En mode brut, les identifiants des personnes d’IMDb et des suppléments sont réunis avant la lecture par blocs. En mode intermédiaire, `name.basics.csv` doit déjà couvrir ces deux sources.

In [7]:
debut = perf_counter()

print("Chargement de Name Basics…")

if UTILISER_INTERIM:
    print("Mode : fichier intermédiaire")
    name_basics = pd.read_csv(
        CHEMIN_NAME_BASICS,
        **OPTIONS_CSV,
    )
else:
    print("Mode : réduction de la source brute")
    ids_personnes = pd.concat([
        title_principals['nconst'],
        supplement_principals['nconst'],
    ]).dropna().unique()
    name_basics = rs.reduire_name_basics(ids_personnes)

print(
    f"[OK] {len(name_basics):,} lignes chargées | "
    f"{perf_counter() - debut:.1f} s"
)

Chargement de Name Basics…
Mode : réduction de la source brute
  Bloc 10 | 2,000,000 lignes lues | 40,225 retenues | 6.8 s
  Bloc 20 | 4,000,000 lignes lues | 41,835 retenues | 13.5 s
  Bloc 30 | 6,000,000 lignes lues | 43,023 retenues | 20.1 s
  Bloc 40 | 8,000,000 lignes lues | 44,090 retenues | 26.7 s
  Bloc 50 | 10,000,000 lignes lues | 46,868 retenues | 33.4 s
  Bloc 60 | 12,000,000 lignes lues | 49,417 retenues | 40.3 s
  Bloc 70 | 14,000,000 lignes lues | 50,427 retenues | 46.8 s
  Bloc 80 | 15,842,044 lignes lues | 50,716 retenues | 53.0 s
  Assemblage des blocs retenus…
  [OK] 15,842,044 lignes lues | 50,716 retenues | 53.0 s
[OK] 50,716 lignes chargées | 53.0 s


### 1.5 — Descriptions françaises

Charger le fichier `film_description_enrichi.csv`. Les descriptions ont été récupérées en français depuis TMDB ou obtenues par traduction ; `source_fr` en conserve la provenance.

In [8]:
descriptions_fr = pd.read_csv(
    CHEMIN_DESCRIPTIONS_FR,
    **OPTIONS_CSV,
)
print(f"[OK] {len(descriptions_fr):,} descriptions chargées")

[OK] 7,356 descriptions chargées


## 2 — Préparation des sources

La préparation sélectionne les colonnes et catégories utiles, résout les doublons et contrôle les correspondances entre les sources avant toute jointure.

### 2.1 — Intégration des suppléments et sélection des personnes

Trier chaque source séparément selon la valeur numérique d’`ordering`, puis concaténer IMDb avant les suppléments. Pour une même relation film-personne-catégorie, la première occurrence est conservée : IMDb reste prioritaire et le plus petit `ordering` est retenu au sein de chaque source.

Seules les quatre colonnes nécessaires à la consolidation des relations et les deux colonnes d’identification des personnes sont conservées.

In [9]:
title_principals_reduit = pd.concat(
    [
        title_principals.sort_values(
            'ordering',
            key=pd.to_numeric,
            kind='stable',
        ),
        supplement_principals.sort_values(
            'ordering',
            key=pd.to_numeric,
            kind='stable',
        ),
    ],
    ignore_index=True,
)

title_principals_reduit = title_principals_reduit[
    ['tconst', 'ordering', 'nconst', 'category']
].copy()
name_basics_reduit = name_basics[['nconst', 'primaryName']].copy()

print(f"Relations après concaténation : {len(title_principals_reduit):,}")
print(f"Personnes chargées : {len(name_basics_reduit):,}")

title_principals_reduit = title_principals_reduit.loc[
    title_principals_reduit['category'].isin(CATEGORIES_RETENUES)
].copy()
title_principals_reduit['ordering'] = pd.to_numeric(
    title_principals_reduit['ordering'],
)
print(f"Relations après sélection des catégories : {len(title_principals_reduit):,}")

avant = len(title_principals_reduit)
title_principals_reduit = title_principals_reduit.drop_duplicates(
    subset=['tconst', 'nconst', 'category'],
)
print(f"Doublons supprimés : {avant - len(title_principals_reduit):,}")
print(f"Relations retenues : {len(title_principals_reduit):,}")

Relations après concaténation : 151,547
Personnes chargées : 50,716
Relations après sélection des catégories : 82,815
Doublons supprimés : 2,064
Relations retenues : 80,751


### 2.2 — Contrôles d’intégrité des personnes

Vérifier les valeurs manquantes et les correspondances entre les identifiants des relations et les noms. Une personne absente de `name.basics` doit être identifiée avant l’agrégation : supprimer un nom seul désalignerait les listes de noms et d’identifiants.

Lors de la préparation du catalogue de référence, ce contrôle a révélé un identifiant erroné dans les suppléments, corrigé dans le fichier source.

In [10]:
print("Valeurs manquantes — Title Principals :")
display(title_principals_reduit.isna().sum())
print("Valeurs manquantes — Name Basics :")
display(name_basics_reduit.isna().sum())

assert not title_principals_reduit.isna().any().any(), (
    "Valeurs manquantes dans les relations film-personne"
)
assert not name_basics_reduit.isna().any().any(), (
    "Valeurs manquantes dans les identifiants ou noms de personnes"
)

masque_absents = ~title_principals_reduit['nconst'].isin(
    name_basics_reduit['nconst'],
)
personnes_absentes = title_principals_reduit.loc[masque_absents]

print(f"Relations sans correspondance : {len(personnes_absentes)}")
print(f"Personnes introuvables : {personnes_absentes['nconst'].nunique()}")
print(f"Films concernés : {personnes_absentes['tconst'].nunique()}")

if not personnes_absentes.empty:
    display(personnes_absentes)

assert personnes_absentes.empty, (
    "Des personnes sont absentes de Name Basics : vérifier les sources "
    "ou régénérer les fichiers intermédiaires."
)

Valeurs manquantes — Title Principals :


tconst      0
ordering    0
nconst      0
category    0
dtype: int64

Valeurs manquantes — Name Basics :


nconst         0
primaryName    0
dtype: int64

Relations sans correspondance : 0
Personnes introuvables : 0
Films concernés : 0


### 2.3 — Préparation des descriptions françaises

Conserver l’identifiant IMDb, la description française et sa provenance.
La description originale reste dans le catalogue initial.
Vérifier l’unicité des identifiants et l’absence de valeurs manquantes
dans le fichier complémentaire.

In [ ]:
descriptions_fr = descriptions_fr[['tconst', 'overview_fr', 'source_fr']].copy()

assert descriptions_fr['tconst'].is_unique, (
    "Identifiants dupliqués dans les descriptions françaises"
)
assert not descriptions_fr.isna().any().any(), (
    "Valeurs manquantes dans les descriptions françaises ou leur provenance"
)

print("[OK] Descriptions françaises validées.")
print("Provenance des descriptions françaises :")
display(descriptions_fr['source_fr'].value_counts())

[OK] Correspondance des descriptions françaises validée.
Provenance des descriptions françaises :


source_fr
TMDB          7286
TRADUCTION      70
Name: count, dtype: int64[pyarrow]

## 3 — Consolidation des personnes par film

Associer les noms aux identifiants IMDb, puis regrouper les personnes selon les deux catégories métier. Une personne ne doit apparaître qu’une fois dans chaque groupe pour un même film ; elle peut toutefois être à la fois actrice et réalisatrice.

### 3.1 — Jointure des relations et des noms

La jointure gauche sur `nconst` conserve toutes les relations sélectionnées. La cardinalité plusieurs-à-un interdit de multiplier les lignes en cas d’identifiants dupliqués dans `name_basics`.

In [12]:
personnes_films = pd.merge(
    left=title_principals_reduit,
    right=name_basics_reduit,
    how='left',
    on='nconst',
    validate='m:1',
)

masque_acteurs = personnes_films['category'].isin(CATEGORIES_ACTEURS)
personnes_films['groupe'] = 'realisateur'
personnes_films.loc[masque_acteurs, 'groupe'] = 'acteur'

personnes_films = personnes_films.drop_duplicates(
    subset=['tconst', 'nconst', 'groupe'],
)

print("Relations retenues par groupe :")
display(personnes_films.groupby('groupe').size())

Relations retenues par groupe :


groupe
acteur         72894
realisateur     7857
dtype: int64

### 3.2 — Distribution du nombre de personnes par film

Compter les personnes distinctes par film et par groupe. Réintégrer les films sans relation pour que les distributions couvrent l’ensemble du catalogue.

Sur le catalogue de référence de 7 356 films, le maximum observé est de dix acteurs et quatre réalisateurs. Les anciennes limites de cinq acteurs et trois réalisateurs ne sont donc pas conservées : toutes les personnes disponibles dans les catégories retenues sont utilisées. `Title Principals` ne constitue pas pour autant une distribution exhaustive des films.

In [13]:
comptes_personnes = (
    personnes_films.groupby(['tconst', 'groupe']).size()
    .unstack(fill_value=0)
    .reindex(index=ids_films, columns=['acteur', 'realisateur'], fill_value=0)
)

print("Distribution du nombre d’acteurs par film :")
display(comptes_personnes['acteur'].value_counts().sort_index())
print("Distribution du nombre de réalisateurs par film :")
display(comptes_personnes['realisateur'].value_counts().sort_index())

Distribution du nombre d’acteurs par film :


acteur
0        5
1       15
2       14
3       11
4        4
5       10
6       13
7       20
8       31
9       44
10    7189
Name: count, dtype: int64

Distribution du nombre de réalisateurs par film :


realisateur
1    6954
2     330
3      45
4      27
Name: count, dtype: int64

### 3.3 — Films sans personne référencée

Afficher les films pour lesquels aucun acteur ou réalisateur n’est retenu. Sur le catalogue de référence, cinq films n’ont pas d’acteur : quatre documentaires et *La Revanche du Frelon Vert*, dont les prestations sont classées en images d’archives dans IMDb. Tous les films disposent d’un réalisateur après intégration des suppléments. Les absences d’acteurs identifiées sont conservées sans imputation.

In [14]:
colonnes_films = ['tconst', 'titre_affiche', 'startYear', 'genre_source']

ids_sans_acteur = comptes_personnes.index[comptes_personnes['acteur'] == 0]
films_sans_acteur = catalogue.loc[
    catalogue['tconst'].isin(ids_sans_acteur),
    colonnes_films,
]
print(f"Films sans acteur : {len(films_sans_acteur):,}")
display(films_sans_acteur)

ids_sans_realisateur = comptes_personnes.index[comptes_personnes['realisateur'] == 0]
films_sans_realisateur = catalogue.loc[
    catalogue['tconst'].isin(ids_sans_realisateur),
    colonnes_films,
]
print(f"Films sans réalisateur : {len(films_sans_realisateur):,}")
if not films_sans_realisateur.empty:
    display(films_sans_realisateur)

Films sans acteur : 5


,tconst,titre_affiche,startYear,genre_source
2965,tt0101376,Atlantis,1991,Documentaire
3995,tt0145519,Le Territoire des autres,1970,Documentaire
4072,tt0162911,La Griffe et la Dent,1976,Documentaire
4213,tt0187039,La Revanche du Frelon Vert,1976,Aventure - Action
5555,tt0478160,Le grand silence,2005,Documentaire


Films sans réalisateur : 0


## 4 — Agrégation des acteurs et réalisateurs

Construire une ligne par film et par groupe. Le tri par identifiant du film puis par `ordering` est effectué avant l’agrégation, afin que les noms et les identifiants soient concaténés dans le même ordre. Aucun nombre maximal de personnes n’est imposé.

### 4.1 — Fonction d’agrégation

`joindre_valeurs()` transforme une série de chaînes en une chaîne unique, avec le séparateur ` | `. Les valeurs manquantes ont déjà été contrôlées ; les films sans personne associée ne sont pas présents dans les groupes à agréger.

In [15]:
def joindre_valeurs(serie):
    return ' | '.join(serie)

### 4.2 — Acteurs principaux

Sélectionner les acteurs, trier les relations, puis agréger les noms et identifiants IMDb dans deux colonnes parallèles.

In [16]:
acteurs_films = personnes_films[personnes_films['groupe'] == 'acteur']
acteurs_films = acteurs_films.sort_values(['tconst', 'ordering'])

acteurs_principaux = acteurs_films.groupby('tconst', as_index=False).agg({
    'primaryName': joindre_valeurs,
    'nconst': joindre_valeurs,
})
acteurs_principaux = acteurs_principaux.rename(columns={
    'primaryName': 'acteurs_principaux',
    'nconst': 'acteurs_nconst',
})

print(f"[OK] {len(acteurs_principaux):,} films avec acteurs")

[OK] 7,351 films avec acteurs


### 4.3 — Réalisateurs

Appliquer la même agrégation aux réalisateurs, en conservant tous les réalisateurs référencés par film.

In [17]:
realisateurs_films = personnes_films[personnes_films['groupe'] == 'realisateur']
realisateurs_films = realisateurs_films.sort_values(['tconst', 'ordering'])

realisateurs_principaux = realisateurs_films.groupby('tconst', as_index=False).agg({
    'primaryName': joindre_valeurs,
    'nconst': joindre_valeurs,
})
realisateurs_principaux = realisateurs_principaux.rename(columns={
    'primaryName': 'realisateurs',
    'nconst': 'realisateurs_nconst',
})

print(f"[OK] {len(realisateurs_principaux):,} films avec réalisateurs")

[OK] 7,356 films avec réalisateurs


## 5 — Enrichissement du catalogue

Ajouter les acteurs, les réalisateurs et les descriptions françaises par jointures gauches sur `tconst`. La cardinalité un-à-un est contrôlée pour chaque jointure. Les films sans acteur ou réalisateur restent dans le catalogue.

### 5.1 — Acteurs et réalisateurs

In [18]:
catalogue_enrichi = pd.merge(
    left=catalogue,
    right=acteurs_principaux,
    how='left',
    on='tconst',
    validate='1:1',
)
catalogue_enrichi = pd.merge(
    left=catalogue_enrichi,
    right=realisateurs_principaux,
    how='left',
    on='tconst',
    validate='1:1',
)

### 5.2 — Descriptions françaises

In [19]:
catalogue_enrichi = pd.merge(
    left=catalogue_enrichi,
    right=descriptions_fr,
    how='left',
    on='tconst',
    validate='1:1',
)

## 6 — Validation du catalogue enrichi

Vérifier l’ajout de six colonnes et la conservation des films, de leur ordre, des valeurs et des types des colonnes initiales. Les absences dans les listes de personnes doivent correspondre aux films identifiés en section 3.3. Les films sans description française conservent des valeurs manquantes dans les colonnes correspondantes.

Les contrôles s’appuient sur le catalogue chargé et les absences effectivement observées, plutôt que sur des effectifs figés. Pour le catalogue de référence, le résultat attendu est de 7 356 films et 24 colonnes, dont cinq films sans acteur et aucun sans réalisateur.

In [ ]:
assert catalogue_enrichi.shape == (len(catalogue), catalogue.shape[1] + 6), (
    "Dimensions incorrectes"
)
assert catalogue_enrichi[catalogue.columns].equals(catalogue), (
    "Le catalogue initial a été modifié"
)

for colonne in ['acteurs_principaux', 'acteurs_nconst']:
    ids_absents = catalogue_enrichi.loc[catalogue_enrichi[colonne].isna(), 'tconst']
    assert set(ids_absents) == set(ids_sans_acteur), (
        f"Absences inattendues : {colonne}"
    )

for colonne in ['realisateurs', 'realisateurs_nconst']:
    ids_absents = catalogue_enrichi.loc[catalogue_enrichi[colonne].isna(), 'tconst']
    assert set(ids_absents) == set(ids_sans_realisateur), (
        f"Absences inattendues : {colonne}"
    )


print("[OK] Catalogue enrichi validé.")
print(f"{catalogue_enrichi.shape[0]:,} films | {catalogue_enrichi.shape[1]} colonnes")
print("Valeurs manquantes dans les colonnes ajoutées :")
display(catalogue_enrichi.iloc[:, catalogue.shape[1]:].isna().sum())

[OK] Catalogue enrichi validé.
7,356 films | 24 colonnes
Valeurs manquantes dans les colonnes ajoutées :


acteurs_principaux     5
acteurs_nconst         5
realisateurs           0
realisateurs_nconst    0
overview_fr            0
source_fr              0
dtype: int64

## 7 — Export du catalogue enrichi

Exporter les colonnes du catalogue initial, suivies des six colonnes d’enrichissement, sans index pandas. Le fichier `data/processed/dataset_catalogue_filtres.csv` est le seul résultat exporté par ce notebook ; `catalogue.csv` reste inchangé.

In [21]:
catalogue_enrichi.to_csv(CHEMIN_CATALOGUE_ENRICHI, index=False)
print(f"[OK] Catalogue enrichi exporté : {CHEMIN_CATALOGUE_ENRICHI}")
print(f"{catalogue_enrichi.shape[0]:,} films | {catalogue_enrichi.shape[1]} colonnes")

[OK] Catalogue enrichi exporté : f:\Antoine\Documents\forward-to-the-past\data\processed\dataset_catalogue_filtres.csv
7,356 films | 24 colonnes
